In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_market_prices

In [2]:
prices = load_market_prices()

prices.head()

Ticker,date,brent_price,wti_price
0,2015-01-02,56.419998,52.689999
1,2015-01-05,53.110001,50.040001
2,2015-01-06,51.099998,47.930000
3,2015-01-07,51.150002,48.650002
4,2015-01-08,50.959999,48.790001


In [3]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935 entries, 0 to 2934
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype        
---  ------       --------------  -----        
 0   date         2935 non-null   datetime64[s]
 1   brent_price  2935 non-null   float64      
 2   wti_price    2934 non-null   float64      
dtypes: datetime64[s](1), float64(2)
memory usage: 68.9 KB


In [4]:
prices.isna().sum()

Ticker
date           0
brent_price    0
wti_price      1
dtype: int64

In [5]:
raw_dir = PROJECT_ROOT / "data" / "raw"

raw_dir.mkdir(
    parents=True,
    exist_ok=True
)

prices.to_csv(
    raw_dir / "market_prices.csv",
    index=False
)

In [8]:
import pandas as pd
def data_quality_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (
            df.isna().mean() * 100
        ).round(2),
        "unique_values": df.nunique()
    })

    return report

In [9]:
data_quality_report(prices)

,dtype,missing,missing_pct,unique_values
Ticker,,,,
date,datetime64[s],0,0.00,2935
brent_price,float64,0,0.00,2346
wti_price,float64,1,0.03,2342


In [11]:
def validate_market_prices(df):

    checks = {
        "negative_wti": (df["wti_price"] < 0).sum(),
        "negative_brent": (df["brent_price"] < 0).sum(),
        "duplicate_dates": df["date"].duplicated().sum(),
        "missing_dates": df["date"].isna().sum()
    }

    return pd.Series(checks)

validate_market_prices(prices)

negative_wti       1
negative_brent     0
duplicate_dates    0
missing_dates      0
dtype: int64